# Experiment 2: Image Augmentation
1) ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05) , val_f1: 0.3682, **test_f1: 0.2551**
2) ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.10) , val_f1 0.3729

## ⚙️ **Libraries Import**

In [395]:
# Set seed for reproducibility
SEED = 100

import numpy as np

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
from pathlib import Path


# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
#from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader
#!pip install torchview
#from torchview import draw_graph

# Configurazione di TensorBoard e directory
#logs_dir = "tensorboard"
#!pkill -f tensorboard
#%load_ext tensorboard
#!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import cv2
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import requests
from io import BytesIO

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

PyTorch version: 2.6.0+cu124
Device: cuda


## 🔧 **Automatic Configuration**

In [396]:
#(LOCAL vs KAGGLE)

def get_paths():
    """It detects the enviroment and returns the correct path."""
    if os.path.exists('/kaggle/input'):
        print("🌍 Detetected enviroment: KAGGLE")
        #!ls "/kaggle/input/dataset"
        input_dir = Path('/kaggle/input/dataset/')
        output_dir = Path('/kaggle/working/')
    else:
        print("💻 Detetected enviroment: LOCAL")
        # Impostate la vostra cartella di input e output locali
        input_dir = Path('../data/archive/')
        output_dir = Path('../data/output/')

        # Crea le cartelle se non esistono in locale
        output_dir.mkdir(parents=True, exist_ok=True)

    return input_dir, output_dir

INPUT_DIR, OUTPUT_DIR = get_paths()
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


🌍 Detetected enviroment: KAGGLE
Input directory: /kaggle/input/dataset
Output directory: /kaggle/working


In [397]:
#creo cartella per i models
models_dir=OUTPUT_DIR/'models'
models_dir.mkdir(parents=True, exist_ok=True)

## **Constants**

In [398]:
#tutte le costanti utili in un unico posto :)

BATCH_SIZE = 64

# ImageNet normalization statistics
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


## ⏳ **Data Loading**

In [399]:
#===============================================
#esc+r TO DEACTIVATE CELL, esc+y TO REACTIVATE IT
#===============================================

def pad_image_with_border_mode(img):
    # VERSIONE COLORE FISSO (#b4b4b4)
    # Padda con un grigio specifico (RGB 180, 180, 180).
    # Ideale se sai che lo sfondo dei vetrini è circa di questo colore.

    h, w = img.shape[:2]
    dim = max(h, w)
    pad_h = (dim - h) // 2
    pad_w = (dim - w) // 2

    top = pad_h
    bottom = dim - h - pad_h
    left = pad_w
    right = dim - w - pad_w

    # Hex #b4b4b4 corrisponde a (180, 180, 180) in RGB
    specific_gray = (180, 180, 180)

    # BORDER_CONSTANT riempie tutto con il valore passato in 'value'
    img_padded = cv2.copyMakeBorder(
        img,
        top, bottom, left, right,
        cv2.BORDER_CONSTANT,
        value=specific_gray
    )

    return img_padded


In [400]:
def normalize_image(img, standardize=True):

    img = img.astype(np.float32)
    # Se i pixel non sono ancora in [0,1], li scala
    if img.max() > 1.0:
        img = img / 255.0

    if standardize:
        # Media e std ImageNet
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std

    return img

In [401]:
#===============================================
#esc+r TO DEACTIVATE CELL, esc+y TO REACTIVATE IT
#===============================================
def load_medical_dataset(folder, target_size=(300, 300), standardize=True):
    """
    Carica immagini e mask da una cartella, applicando padding, resize e            normalizzazione.
    """

    images = []
    masks = []

    if not os.path.exists(folder):
        print(f"Error: Folder {folder} does not exist.")
        return np.array([]), np.array([])

    img_files = sorted([f for f in os.listdir(folder) if f.startswith('img_')
    and f.endswith('.png')])
    print(f"Found {len(img_files)} images in {folder}")

    for img_file in img_files:
        mask_file = img_file.replace('img_', 'mask_')
        img_path = os.path.join(folder, img_file)
        mask_path = os.path.join(folder, mask_file)

        if not os.path.exists(mask_path):
            print(f"Warning: Mask not found for {img_file}, skipping...")
            continue

        # Carica immagine e mask
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            print(f"Warning: Could not load {img_file} or {mask_file}, skipping...")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 1️⃣ Padding veloce
        img = pad_image_with_border_mode(img)

        # 2️⃣ Resize
        img = cv2.resize(img, target_size)

        # 3️⃣ Normalizzazione / standardizzazione
        img = normalize_image(img, standardize=standardize)

        # 4️⃣ Gestione mask: padding nero, resize, [0,1]
        h, w = mask.shape[:2]
        dim = max(h, w)
        pad_h = (dim - h) // 2
        pad_w = (dim - w) // 2
        mask = np.pad(mask, ((pad_h, dim - h - pad_h), (pad_w, dim - w - pad_w)),
                      mode='constant', constant_values=0)
        mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=-1)

        images.append(img)
        masks.append(mask)

    print(f"Successfully loaded {len(images)} image-mask pairs")
    return np.array(images), np.array(masks)

In [402]:
train_path = INPUT_DIR/'train_data'
test_path = INPUT_DIR/'test_data'

train_images, train_masks = load_medical_dataset(train_path, standardize=False)
test_images, test_masks = load_medical_dataset(test_path, standardize=False)

Found 629 images in /kaggle/input/dataset/train_data
Successfully loaded 629 image-mask pairs
Found 477 images in /kaggle/input/dataset/test_data


KeyboardInterrupt: 

In [ ]:


def remove_green_artifacts(imgs):
    """
    Rimuove le macchie verdi dalle immagini impostando i pixel a grigio (#b4b4b4).
    Accetta in input un array numpy (N, C, H, W) o (N, H, W, C).
    """
    print("--- REMOVING GREEN ARTIFACTS ---")
    
    # PARAMETRI PER IL VERDE
    SOGLIA_VERDE_MIN = 0.2
    DIFFERENZA_DOMINANTE = 0.05
    DILATION_KERNEL_SIZE = 3
    ITERAZIONI_DILATAZIONE = 2 
    
    # COLORE DI RIEMPIMENTO (Grigio #b4b4b4)
    # Normalizzato tra 0 e 1 se le immagini sono float, o 0-255 se uint8
    # Assumiamo float [0,1] dato che poi fai astype('float32')
    FILL_COLOR = 0.70588  # 180/255 = 0.70588... (#b4b4b4)

    cleaned_imgs = imgs.copy()

    # --- RILEVAMENTO FORMATO ---
    if imgs.shape[1] == 3: # (N, 3, H, W)
        # print("Formato: Channels First")
        R = imgs[:, 0, :, :]
        G = imgs[:, 1, :, :]
        B = imgs[:, 2, :, :]
        axis_to_expand = 1
    else: # (N, H, W, 3)
        # print("Formato: Channels Last")
        R = imgs[:, :, :, 0]
        G = imgs[:, :, :, 1]
        B = imgs[:, :, :, 2]
        axis_to_expand = -1

    # 1. Trova i pixel sicuramente verdi (Core)
    green_mask_bool = (G > (R + DIFFERENZA_DOMINANTE)) & \
                      (G > (B + DIFFERENZA_DOMINANTE)) & \
                      (G > SOGLIA_VERDE_MIN)

    print(f"Green pixels detected: {np.sum(green_mask_bool)}")

    # Convertiamo boolean in uint8 per OpenCV (0 e 255)
    mask_uint8 = (green_mask_bool * 255).astype(np.uint8)

    # Creiamo il kernel 
    kernel = np.ones((DILATION_KERNEL_SIZE, DILATION_KERNEL_SIZE), np.uint8)

    final_green_mask = np.zeros_like(green_mask_bool)

    # Iteriamo per la dilatazione (OpenCV non supporta batch 3D diretti per dilate)
    for i in range(len(mask_uint8)):
        dilated = cv2.dilate(mask_uint8[i], kernel, iterations=ITERAZIONI_DILATAZIONE)
        final_green_mask[i] = dilated > 0

    # --- APPLICAZIONE ALL'IMMAGINE ---
    # Espandiamo la maschera per il broadcasting
    green_mask_expanded = np.expand_dims(final_green_mask, axis=axis_to_expand)
    
    # FIX: Usiamo np.where per sostituire i pixel verdi con il colore di riempimento
    # np.where(condition, x, y) -> Se condition è True prendi x, altrimenti y
    # Se è verde (green_mask_expanded == True), metti FILL_COLOR, altrimenti lascia l'immagine originale
    cleaned_imgs = np.where(green_mask_expanded, FILL_COLOR, cleaned_imgs)
    
    print("Green artifacts removed (replaced with gray #b4b4b4).")
    return cleaned_imgs

In [ ]:
# I numpy sono nel dataset ora (MESSI SUCCESSIVAMENTE)

#train_images = np.load(INPUT_DIR/'train_images.npy')
#train_masks = np.load(INPUT_DIR/'masks_cleaned.npy')
#test_images = np.load(INPUT_DIR/'test_images.npy')
#test_masks = np.load(INPUT_DIR/'test_masks.npy')
train_labels = pd.read_csv(INPUT_DIR/'train_labels.csv')

train_images = remove_green_artifacts(train_images)

train_images=train_images.astype('float32')
train_masks=train_masks.astype('float32')
test_images=test_images.astype('float32')
test_masks=test_masks.astype('float32')

print(type(train_images))

# Get class names
class_names = train_labels['label']
#print(class_names)

print("Data loaded sucessfully!")
print(f"Train images shape: {train_images.shape}")
print(f"Train masks shape: {train_masks.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Test masks shape: {test_masks.shape}")
print(f"Train labels shape: {train_labels.shape}")

## 🔎 **Data Analysis**

In [ ]:
#LABELS ANALYSIS AND DISTRIBUTION
print(train_labels.describe())

print("\n--- Unique Labels ---")
unique_labels = train_labels['label'].unique()
print(unique_labels)

print("\n--- Label Distribution ---")
label_counts = train_labels['label'].value_counts()
label_percentages = train_labels['label'].value_counts(normalize=True) * 100
label_stats = pd.DataFrame({
    'count': label_counts,
    'percentage': label_percentages
})
print(label_stats.round(2))

In [ ]:
#number of classes
NUM_CLASSES = len(unique_labels)

## 🔄 **Data Processing**

In [ ]:
# Target preparation

# Define the mapping between textual labels and numerical value (enum)
label_mapping = {
    'Luminal A': 0,
    'Luminal B': 1,
    'HER2(+)': 2,
    'Triple negative': 3
}

# Add new column 'target' to train_labels dataframe
train_labels['target'] = train_labels['label'].map(label_mapping)
target = np.array(train_labels['target']).astype('float32')

# weight = (num_total_training / (num_classes * num_for_class))
# Reindex label_stats to follow the explicit mapping order instead of alphabetical order
ordered_labels = [label for label, _ in sorted(label_mapping.items(), key=lambda kv: kv[1])]
label_stats_ordered = label_stats.reindex(ordered_labels)
class_weights = (len(train_labels) / (NUM_CLASSES * label_stats_ordered['count'])).astype(np.float32)

# Create list useful later for tensor transformation
class_weights_list = list(class_weights)
print(f"Balanced WEIGHT for each class:\t{class_weights_list}")
print(f"Class order: {ordered_labels}")
class_weights_tensor = torch.tensor(class_weights_list, dtype=torch.float32).to(device)

In [ ]:
# SPLIT in TRAIN & VAL

X_train , X_val, y_train, y_val, mask_train, mask_val = train_test_split(train_images, target, train_masks, test_size=0.2, random_state=SEED,  stratify=target)

# Print the shapes of the resulting datasets
print("Training Data Shape:", X_train.shape)
print("Training Label Shape:", y_train.shape)
print("Training Mask Shape:", mask_train.shape)
print("Validation Data Shape:", X_val.shape)
print("Validation Label Shape:", y_val.shape)
print("Validation Mask Shape:", mask_val.shape)

In [ ]:
# Define the input shape based on the training data
INPUT_SHAPE = (X_train.shape[3], X_train.shape[1], X_train.shape[2])
print(INPUT_SHAPE)

In [ ]:
class MyDataset(torch.utils.data.Dataset):
    """Custom PyTorch Dataset with optional augmentation and normalization."""

    def __init__(self, data, labels, masks=None, augmentation=None, normalize_imagenet=False):
        """
        Args:
            data: numpy array of images (N, C, H, W) in range [0, 1]
            labels: numpy array of labels
            masks: numpy array of masks (N, H, W) or (N, H, W, 1)
            augmentation: transforms for data augmentation
            normalize_imagenet: whether to apply ImageNet normalization
        """
        self.data = torch.from_numpy(data).permute(0, 3, 1, 2)
        
        # Normalize images to [0, 1] if they are in [0, 255]
        if self.data.max() > 1.0:
            self.data = self.data / 255.0
            
        self.labels = torch.from_numpy(labels).squeeze().long()
        self.augmentation = augmentation
        self.normalize_imagenet = normalize_imagenet
        
        if masks is not None:
            # Normalize masks to [0, 1] if needed
            if masks.max() > 1.0:
                masks = masks / 255.0
            
            # Ensure masks are (N, 1, H, W)
            if masks.ndim == 3: # (N, H, W)
                masks = np.expand_dims(masks, axis=-1)
            
            self.masks = torch.from_numpy(masks).permute(0, 3, 1, 2)
        else:
            self.masks = None

        # ImageNet normalization transform
        if normalize_imagenet:
            self.normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        else:
            self.normalize = None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx].clone()  # Clone to avoid modifying original
        label = self.labels[idx]
        
        # Apply augmentation if provided
        if self.augmentation:
            if self.masks is not None:
                # CRITICAL: Pass tuple (image, mask) to augmentation pipeline
                # The first transform (ApplyMask) MUST be designed to handle this tuple 
                # and return just the image tensor for subsequent transforms.
                mask = self.masks[idx].clone()
                image = self.augmentation((image, mask))
            else:
                image = self.augmentation(image)

        # Apply ImageNet normalization if requested
        if self.normalize:
            image = self.normalize(image)

        return image, label

## 🎨 Image Augmentation Classses 


In [ ]:
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image


class ZoomToMaskROI(nn.Module):
    """Zooms on the mask foreground and keeps tuples for downstream transforms."""

    def __init__(self, padding_ratio=0.15, min_padding=6, p=1.0):
        super().__init__()
        self.padding_ratio = padding_ratio
        self.min_padding = min_padding
        self.p = p

    def forward(self, input_data):
        if not (isinstance(input_data, (tuple, list)) and len(input_data) == 2):
            return input_data
        if random.random() > self.p:
            return input_data

        img, mask = input_data
        if mask is None:
            return input_data

        if mask.dim() == 3:
            mask_2d = mask[0] if mask.size(0) == 1 else mask.mean(dim=0)
        elif mask.dim() == 2:
            mask_2d = mask
        else:
            return input_data

        binary_mask = (mask_2d > 0.5)
        if binary_mask.sum() == 0:
            return input_data

        coords = torch.nonzero(binary_mask, as_tuple=False)
        y_min = int(coords[:, 0].min())
        y_max = int(coords[:, 0].max())
        x_min = int(coords[:, 1].min())
        x_max = int(coords[:, 1].max())

        H, W = mask_2d.shape
        box_h = y_max - y_min + 1
        box_w = x_max - x_min + 1
        pad_h = max(int(box_h * self.padding_ratio), self.min_padding)
        pad_w = max(int(box_w * self.padding_ratio), self.min_padding)

        y0 = max(0, y_min - pad_h)
        y1 = min(H, y_max + pad_h + 1)
        x0 = max(0, x_min - pad_w)
        x1 = min(W, x_max + pad_w + 1)

        if (y1 - y0) < 2 or (x1 - x0) < 2:
            return input_data

        roi_img = img[:, y0:y1, x0:x1].unsqueeze(0)
        zoomed_img = F.interpolate(
            roi_img,
            size=(img.shape[1], img.shape[2]),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        mask_dtype = mask.dtype
        target_size = (mask.shape[-2], mask.shape[-1])

        if mask.dim() == 3:
            roi_mask = mask[:, y0:y1, x0:x1].unsqueeze(0)
            zoomed_mask = F.interpolate(roi_mask, size=target_size, mode="nearest").squeeze(0)
        else:
            roi_mask = mask[y0:y1, x0:x1].unsqueeze(0).unsqueeze(0)
            zoomed_mask = F.interpolate(roi_mask, size=target_size, mode="nearest").squeeze(0).squeeze(0)

        zoomed_mask = zoomed_mask.to(mask_dtype)
        return zoomed_img, zoomed_mask


class ApplyMask(nn.Module):
    """
    Applies the binary mask to the image with probability p.
    Expects a tuple (image, mask) as input.
    Returns the image (Tensor).
    """
    def __init__(self, p=0.5, fill_color=0.0, color_variation=0.05):
        super().__init__()
        self.p = p
        self.fill_color = fill_color
        self.color_variation = color_variation

    def forward(self, input_data):
        # Handle tuple input (image, mask)
        if isinstance(input_data, (tuple, list)) and len(input_data) == 2:
            img, mask = input_data
            
            # Apply mask with probability p
            if random.random() <= self.p:
                # Ensure mask is binary (0 or 1)
                mask = (mask > 0.5).float()
                
                # Generate random fill color around fill_color
                # Create a random color tensor of shape (C, 1, 1) to broadcast
                C = img.shape[0]
                # Random variation in range [-color_variation, +color_variation]
                noise = (torch.rand(C, 1, 1, device=img.device) * 2 - 1) * self.color_variation
                
                current_fill = self.fill_color + noise
                current_fill = torch.clamp(current_fill, 0.0, 1.0)

                # Apply mask: keep image where mask is 1, use current_fill where mask is 0
                img = img * mask + current_fill * (1 - mask)
            
            return img # Return only the image for subsequent transforms
        
        # If input is just image (e.g. validation or no mask provided), return it as is
        return input_data

class AddGreenSpot(nn.Module):
    """
    Adds a green spot artifact to the image. 
    Loads the spot shape from a file.
    """
    def __init__(self, spot_path, size=(20, 20), p=0.5):
        super().__init__()
        self.size = size
        self.p = p
        
        self.spot_tensor = None
        
        if os.path.exists(spot_path):
            try:
                # Load image and convert to RGBA (for transparency)
                spot_pil = Image.open(spot_path).convert("RGBA")
                spot_pil = spot_pil.resize(size)
                self.spot_tensor = transforms.ToTensor()(spot_pil) # (4, H, W) in range [0, 1]
                print(f"✅ Loaded custom spot from {spot_path}")
            except Exception as e:
                print(f"⚠️ Error loading spot image: {e}")
                raise e # Raise error if image loading fails
        else:
             raise FileNotFoundError(f"Spot image not found at {spot_path}")


    def forward(self, img):
        # Apply with probability p
        if random.random() > self.p:
            return img

        # img: Tensor (C, H, W)
        _, H, W = img.shape
        
        # 1. Randomly rotate the spot
        angle = random.uniform(0, 360)
        rotated_spot = TF.rotate(self.spot_tensor, angle, fill=[0.0])
        
        spot_rgb = rotated_spot[:3]
        spot_alpha = rotated_spot[3:4] 
        
        # 2. Overlay at RANDOM position
        # Use actual tensor dimensions to avoid shape mismatch with PIL resize
        _, h, w = self.spot_tensor.shape
        
        # Generate random coordinates ensuring the spot is within the image boundaries
        max_x = max(0, W - w)
        max_y = max(0, H - h)
        
        x = random.randint(0, max_x)
        y = random.randint(0, max_y)
        
        # Calculate bounds
        y_end = min(y + h, H)
        x_end = min(x + w, W)
        h_eff = y_end - y
        w_eff = x_end - x
        
        if h_eff > 0 and w_eff > 0:
            spot_rgb_crop = spot_rgb[:, :h_eff, :w_eff]
            spot_alpha_crop = spot_alpha[:, :h_eff, :w_eff]
            
            roi = img[:, y:y_end, x:x_end]
            
            # Determine max value (1.0 or 255.0) based on image content
            max_val = 1.0 if img.max() <= 1.0 else 255.0
            
            # Move to device
            spot_rgb_crop = spot_rgb_crop.to(img.device)
            spot_alpha_crop = spot_alpha_crop.to(img.device)
            
            # Alpha blending
            img[:, y:y_end, x:x_end] = (spot_rgb_crop * max_val * spot_alpha_crop) + (roi * (1 - spot_alpha_crop))
                
        return img

class HEDJitter(nn.Module):
    """
    Randomly perturbs Hematoxylin & Eosin (H&E) stain concentrations.
    """
    def __init__(self, alpha_range=(0.95, 1.05), beta_range=(-0.05, 0.05), p=0.5):
        super().__init__()
        self.alpha_range = alpha_range
        self.beta_range = beta_range
        self.p = p
        
        # Ruifrok & Johnston H&E stain reference
        stain_ref = torch.tensor([
            [0.650, 0.704, 0.286],  # Hematoxylin
            [0.072, 0.990, 0.105],  # Eosin
            [0.268, 0.571, 0.766]   # DAB
        ], dtype=torch.float32)
        
        # Normalize rows
        stain_ref = stain_ref / torch.norm(stain_ref, dim=1, keepdim=True)
        
        # Register stain matrices as buffers
        self.register_buffer('stain_mat', stain_ref)
        self.register_buffer('stain_mat_inv', torch.linalg.inv(stain_ref))

    def forward(self, img):
        """
        img: Tensor of shape (C, H, W) in range [0, 1]
        """
        if random.random() > self.p or img.shape[0] != 3:
            return img  # Skip augmentation

        device = img.device
        img_orig = img.clone()
        
        # 1. Convert to Optical Density (OD)
        epsilon = 1e-6
        img_clamped = torch.clamp(img, epsilon, 1.0)
        od = -torch.log(img_clamped) / 2.302585  # Convert ln to log10

        # 2. Create tissue mask
        tissue_mask = (od.std(dim=0) > 0.02).float().unsqueeze(0)

        # 3. Deconvolve to stain concentrations
        C, H, W = od.shape
        od_flat = od.permute(1, 2, 0).reshape(-1, 3)
        concentrations = torch.matmul(od_flat, self.stain_mat_inv.T)

        # 4. Apply random augmentation
        alpha = torch.empty(1, 3, device=device).uniform_(*self.alpha_range)
        beta = torch.empty(1, 3, device=device).uniform_(*self.beta_range)
        concentrations = torch.clamp(concentrations * alpha + beta, 0, None)

        # 5. Reconstruct OD and convert back to RGB
        od_recon = torch.matmul(concentrations, self.stain_mat)
        od_recon = od_recon.reshape(H, W, C).permute(2, 0, 1)
        img_aug = torch.pow(10, -od_recon)
        img_aug = torch.clamp(img_aug, 0, 1)

        # 6. Blend augmented image with original using tissue mask
        return img_aug * tissue_mask + img_orig * (1 - tissue_mask)

fill_color = 180.0 / 255.0
k_cutout = 5

cutouts = [
    transforms.RandomErasing(p=0.2, scale=(0.01, 0.033), value=0)
    for _ in range(k_cutout)
]

train_transforms = transforms.Compose([
    ApplyMask(p=0.2, fill_color=fill_color, color_variation=0.05), # Apply mask first (handles tuple input)
    ZoomToMaskROI(padding_ratio=0.15, min_padding=6, p=0.2),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), fill=fill_color),
    transforms.RandomVerticalFlip(p=0.3),
    #AddGreenSpot(spot_path="/kaggle/input/dataset/green_spot.png", size=(84, 63), p=0.3), 
    transforms.RandomApply([transforms.RandomRotation(15, fill=fill_color)], p=0.3),
    #transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))], p=0.3),
    HEDJitter(alpha_range=(0.95, 1.05), beta_range=(-0.01, 0.01), p=0.3),
]+cutouts
                                     )
# Convert numpy arrays to customized PyTorch datasets
# Pass mask_train to the dataset
train_ds = MyDataset(X_train, y_train, masks=mask_train, augmentation=train_transforms, normalize_imagenet=True)
val_ds   = MyDataset(X_val, y_val, masks=None, augmentation=None, normalize_imagenet=True)

In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [ ]:
# Create data loaders with different settings for each phase
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
# print batch shape
for xb, yb in train_loader:
    print("Features batch shape:", xb.shape)
    print("Labels batch shape:", yb.shape)
    break # Stop after getting one batch

In [ ]:
# Visualize some augmented images with ZOOM (Smart Cropping)
def denormalize(tensor):
    """Reverses ImageNet normalization for visualization."""
    tensor = tensor.clone()
    mean = torch.tensor(IMAGENET_MEAN, dtype=torch.float32, device=tensor.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=torch.float32, device=tensor.device).view(3, 1, 1)
    tensor.mul_(std).add_(mean)
    return tensor

def crop_to_content(img_tensor, pad=20):
    """Crops the image to the bounding box of the non-uniform content (tissue)."""
    # Convert to numpy (H, W, C)
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = np.clip(img_np, 0, 1)
    img_uint8 = (img_np * 255).astype(np.uint8)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # Edge detection to find texture/tissue
    # Canny is good for finding the boundary between flat background and textured tissue
    edges = cv2.Canny(gray, 50, 150)
    
    # Dilate to connect regions and fill small gaps
    kernel = np.ones((5,5), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=4)
    
    # Find contours
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return img_tensor # Return original if no content found
        
    # Find bounding box covering all contours
    x_min, y_min = np.inf, np.inf
    x_max, y_max = -np.inf, -np.inf
    
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        x_min = min(x_min, x)
        y_min = min(y_min, y)
        x_max = max(x_max, x + w)
        y_max = max(y_max, y + h)
        
    # Apply padding
    H, W = gray.shape
    x_min = int(max(0, x_min - pad))
    y_min = int(max(0, y_min - pad))
    x_max = int(min(W, x_max + pad))
    y_max = int(min(H, y_max + pad))
    
    # Check if crop is valid
    if x_max <= x_min or y_max <= y_min:
        return img_tensor

    # Crop
    return img_tensor[:, y_min:y_max, x_min:x_max]

# Get a batch of images
images, labels = next(iter(train_loader))

# Plot a few examples
plt.figure(figsize=(20, 10))
for i in range(8):  # Show 8 examples
    ax = plt.subplot(2, 4, i + 1)
    
    # Denormalize
    img_tensor = denormalize(images[i])
    
    # ZOOM: Crop to content
    img_tensor_cropped = crop_to_content(img_tensor)
    
    # Convert to numpy for display
    img = img_tensor_cropped.permute(1, 2, 0).cpu().numpy()
    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    
    # Get label name
    label_idx = labels[i].item()
    label_name = ordered_labels[label_idx]
    
    plt.title(f"Label: {label_name}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 🧮 **Network Parameters**

In [ ]:
# Training parameters
LEARNING_RATE_HEAD = 0.001
LEARNING_RATE_CNN = 0.0001
EPOCHS = 200
PATIENCE = 25


# Regularisation
DROPOUT_RATE = 0.4      # Dropout probability
#L1_LAMBDA = 1e-3        # L1 penalty NON È USATA PER ORA
L2_HEAD =  1e-3       # L2 penalty NON È USATA PER ORA
L2_CNN = 1e-4

# Set up loss function
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

##  🧠 **Training Functions**

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device):
    """Train for one epoch."""
    model.train()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='macro'
    )

    return epoch_loss, epoch_f1


def validate_one_epoch(model, val_loader, criterion, device):
    """Validate for one epoch."""
    model.eval()

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='macro'
    )

    return epoch_loss, epoch_f1


def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        scheduler=None, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, verbose=1, experiment_name=""):
    """Train the neural network model."""

    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device
        )

        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Scheduler step (moved after validation for ReduceLROnPlateau)
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1={val_f1:.4f}")

        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), OUTPUT_DIR/'models'/(experiment_name+'_model.pt'))
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break

    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load(OUTPUT_DIR/'models'/(experiment_name+'_model.pt')))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    if patience == 0:
        torch.save(model.state_dict(), OUTPUT_DIR/'models'/(experiment_name+'_model.pt'))

    return model, training_history

## 🛠️ **Transfer Learning**

In [ ]:
class ModelTransferLearning(nn.Module):
    """
    CNN model with pretrained weights for transfer learning.

    IMPORTANT: This model expects ImageNet-normalized inputs!
    Images should be normalized with mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    """
    def __init__(self, model_name, num_classes, dropout_rate=0.3, freeze_backbone=False):
        super().__init__()
        
        # 1. Load the requested backbone
        if model_name == "resnet18":
            self.backbone = torchvision.models.resnet18(weights='IMAGENET1K_V1')
            num_features = self.backbone.fc.in_features # 512
            
        elif model_name == "resnet50":
            self.backbone = torchvision.models.resnet50(weights='IMAGENET1K_V2')
            num_features = self.backbone.fc.in_features # 2048
            
        elif model_name == "efficientnet_b0":
            self.backbone = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
            num_features = self.backbone.classifier[1].in_features # 1280
            self.backbone.classifier = nn.Identity() 
            
        elif model_name == "efficientnet_b3":
            self.backbone = torchvision.models.efficientnet_b3(weights='IMAGENET1K_V1')
            num_features = self.backbone.classifier[1].in_features # 1792
            self.backbone.classifier = nn.Identity()

        elif model_name == "densenet121":
            self.backbone = torchvision.models.densenet121(weights='IMAGENET1K_V1')
            num_features = self.backbone.classifier.in_features # 1024
            self.backbone.classifier = nn.Identity()

        elif model_name == "mobilenet_v3_small":
            self.backbone = torchvision.models.mobilenet_v3_small(weights='IMAGENET1K_V1')
            num_features = self.backbone.classifier[0].in_features # 576
            self.backbone.classifier = nn.Identity()

        else:
            raise ValueError(f"Unknown model name: {model_name}")

        # 2. Freeze/Unfreeze Backbone
        for param in self.backbone.parameters():
            param.requires_grad = not freeze_backbone
        
        for param in self.backbone.features[-3:].parameters():
            param.requires_grad = True   # ultimi 3 blocchi addestrabili

        # 3. Create a common Classifier Head
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(num_features, 512),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(512),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 256),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(256),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )
        
        # 4. Handle Architecture Differences (ResNet uses .fc, others use .classifier)
        if "resnet" in model_name:
            self.backbone.fc = self.head
        else:
            pass  # backbone is feature extractor, head applied in forward

    def forward(self, x):
        if "resnet" in self.backbone.__class__.__name__.lower():
            return self.backbone(x)
        
        x = self.backbone(x)
        return self.head(x)


In [ ]:
# Choose your model 
# Options: "resnet18", "resnet50", "efficientnet_b0", "mobilenet_v3_small"
MODEL_NAME = "efficientnet_b3"

# Create transfer learning model
tl_model = ModelTransferLearning(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, freeze_backbone=True).to(device)

# Display model summary
summary(tl_model, input_size=INPUT_SHAPE)

In [ ]:
# Setup training
EXPERIMENT_NAME = "image_augmentation"

#optimizer = torch.optim.AdamW(tl_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)
optimizer = torch.optim.AdamW([
    {'params': tl_model.backbone.features[-3:].parameters(), 'lr': LEARNING_RATE_CNN, 'weight_decay': L2_CNN},
    {'params': tl_model.head.parameters(), 'lr': LEARNING_RATE_HEAD, 'weight_decay': L2_HEAD}
])
#optimizer = torch.optim.Adam(tl_model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

# Scheduler: ReduceLROnPlateau
# Reduces learning rate when a metric has stopped improving.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

In [ ]:
%%time

# Train with transfer learning
tl_model, tl_history = fit(
    model=tl_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    scheduler=scheduler, # Pass the scheduler here
    verbose=1,
    experiment_name=EXPERIMENT_NAME,
    patience=PATIENCE
)

final_val_f1 = round(max(tl_history['val_f1']) * 100, 2)
print(f'Final validation F1: {final_val_f1}%')

In [ ]:
# @title Plot History
# Create a figure with two side-by-side subplots
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(tl_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(tl_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Categorical Crossentropy Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation Accuracy on the second axis
ax2.plot(tl_history['train_f1'], label='Training F1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(tl_history['val_f1'], label='Validation F1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# @title Plot Confusion Matrix
# Collect predictions and ground truth labels
class_labels = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']

val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb, yb in val_loader:
        xb = xb.to(device)

        # Forward pass: get model predictions
        logits = tl_model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Store accuracy per class with desired names
#Calcola il totale per ogni classe reale (somma delle righe)
true_totals = cm.sum(axis=1)
#Normalizza la matrice (gestendo la divisione per zero)
cm_perc = np.divide(cm.astype(float), true_totals[:, None], out=np.zeros_like(cm, dtype=float), where=true_totals[:, None]>0)
#Estrai la diagonale (che è l'accuratezza per ogni classe)
per_class_accuracy = cm_perc.diagonal()

accuracy_LA = per_class_accuracy[0]
accuracy_LB = per_class_accuracy[1]
accuracy_H2 = per_class_accuracy[2]
accuracy_TN = per_class_accuracy[3]

print("-" * 35)
print(f"{'CLASS':<20} {'ACCURACY'}")
print("-" * 35)

print(f"{'Luminal A':<20} {accuracy_LA:.2%}")
print(f"{'Luminal B':<20} {accuracy_LB:.2%}")
print(f"{'HER2(+)':<20} {accuracy_H2:.2%}")
print(f"{'Triple negative':<20} {accuracy_TN:.2%}")
print("-" * 35)

# Visualise confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels
            )
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

## 🕹️ **Test Inference**

In [ ]:
#PREPARE DATA FOR TESTING
test_ds = MyDataset(test_images, np.zeros(len(test_images)), augmentation=None, normalize_imagenet=True)
test_loader = make_loader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
# Load the  model for inference
best_model = ModelTransferLearning(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, freeze_backbone=True).to(device)
best_model.load_state_dict(torch.load(OUTPUT_DIR/"models"/(EXPERIMENT_NAME+"_model.pt")))
best_model.eval()

print("Model loaded successfully!")

In [ ]:
# Collect predictions and ground truth labels
test_preds = []
with torch.no_grad():  # Disable gradient computation for inference
    for batch in test_loader:

        xb = batch[0].to(device)

        # Forward pass: get model predictions
        logits = best_model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        test_preds.append(preds)

# Combine all batches into single arrays
test_preds = np.concatenate(test_preds)

# Create CSV SUBMISSION
target_to_label = {
    0: 'Luminal A',
    1: 'Luminal B',
    2: 'HER2(+)',
    3: 'Triple negative'
}

# Ricostruisco il nome delle immagini: i=0 -> "img_0000.png", i=10 -> "img_0010.png"
filenames = [f"img_{i:04d}.png" for i in range(len(test_preds))]
predicted_labels = [target_to_label[p] for p in test_preds]

df_results = pd.DataFrame({
    "sample_index": filenames,
    "label": predicted_labels
})

csv_path = OUTPUT_DIR / 'test_predictions.csv'
df_results.to_csv(csv_path, index=False, header=True)
print(f"✅ test_predictions.csv created and saved successfully in: {csv_path}")